In [3]:
import tensorflow as tf
import tensorflow_data_validation as tfdv
import pandas as pd
import numpy as np
from tensorflow_metadata.proto.v0 import schema_pb2

print('Tf version: ', tf.__version__)
print('TFDV version: ', tfdv.__version__)

Tf version:  2.17.0
TFDV version:  1.14.0


In [107]:
# -------------------------
# 📂 Load raw data
# -------------------------
raw_df = pd.read_csv('dataset/laptop_data_1M_v2_CLEANED.csv')
print(f"Loaded {len(raw_df):,} rows.")

Loaded 1,000,000 rows.


In [113]:
# Select numeric columns
numeric_df = raw_df.select_dtypes(include=[np.number])

summary = pd.DataFrame({
    'min': numeric_df.min(),
    'max': numeric_df.max(),
    'mean': numeric_df.mean(),
    'median': numeric_df.median(),
    #'mode': numeric_df.mode()
})
print(summary)

             min       max          mean    median
Inches   10.1000      35.6     15.143428     15.60
Ram       1.0000      64.0      8.467997      8.00
Weight    0.0002      11.1      2.081096      2.06
Price  -500.0000  999999.0  62455.164907  52054.56


In [114]:
# -------------------------
# 🔀 Split into train, eval, test
# -------------------------
train_df = raw_df.sample(frac=0.8, random_state=42)
remaining_df = raw_df.drop(train_df.index)
eval_df = remaining_df.sample(frac=0.5, random_state=42)  # 10% of total
test_df = remaining_df.drop(eval_df.index)                 # 10% of total

print(
    f"Train: {len(train_df)}, Eval: {len(eval_df)}, Test: {len(test_df)}"
)


Train: 800000, Eval: 100000, Test: 100000


In [115]:
# Start with an empty schema
schema = schema_pb2.Schema()

# Define Price as float with value range
price_feature = schema.feature.add()
price_feature.name = 'Price'
price_feature.type = schema_pb2.FeatureType.FLOAT
price_domain = price_feature.float_domain
price_domain.min = 10000.0
price_domain.max = 300000.0

In [116]:
# Define Company as string
company_feature = schema.feature.add()
company_feature.name = 'Company'
company_feature.type = schema_pb2.FeatureType.BYTES

In [117]:
# Define TypeName as string
typename_feature = schema.feature.add()
typename_feature.name = 'TypeName'
typename_feature.type = schema_pb2.FeatureType.BYTES

In [118]:
# Define Inches as float
inches_feature = schema.feature.add()
inches_feature.name = 'Inches'
inches_feature.type = schema_pb2.FeatureType.FLOAT
inches_domain = inches_feature.float_domain
inches_domain.min = 10.0
inches_domain.max = 32.0

In [119]:
# Define ScreenResolution as string
screen_resolution_feature = schema.feature.add()
screen_resolution_feature.name = 'ScreenResolution'
screen_resolution_feature.type = schema_pb2.FeatureType.BYTES

In [120]:
# Define Ram as float
ram_feature = schema.feature.add()
ram_feature.name = 'Ram'
ram_feature.type = schema_pb2.FeatureType.FLOAT
ram_domain = ram_feature.float_domain
ram_domain.min = 1.0
ram_domain.max = 64.0

In [121]:
# Define Memory as string
memory_feature = schema.feature.add()
memory_feature.name = 'Memory'
memory_feature.type = schema_pb2.FeatureType.BYTES

In [122]:
# Define OpSys as string
opsys_feature = schema.feature.add()
opsys_feature.name = 'OpSys'
opsys_feature.type = schema_pb2.FeatureType.BYTES

In [123]:
# Define Weight as string
weight_feature = schema.feature.add()
weight_feature.name = 'Weight'
weight_feature.type = schema_pb2.FeatureType.FLOAT
weight_domain = weight_feature.float_domain
weight_domain.min = 0.5
weight_domain.max = 5.0

In [124]:
# Define Cpu as string
cpu_feature = schema.feature.add()
cpu_feature.name = 'Cpu'
cpu_feature.type = schema_pb2.FeatureType.BYTES

In [125]:
# Define Gpu as string
gpu_feature = schema.feature.add()
gpu_feature.name = 'Gpu'
gpu_feature.type = schema_pb2.FeatureType.BYTES

In [126]:
tfdv.write_schema_text(schema, 'manual_schema.pbtxt')

In [127]:
# Read manual schema
schema = tfdv.load_schema_text('manual_schema.pbtxt')
schema

feature {
  name: "Price"
  type: FLOAT
  float_domain {
    min: 10000.0
    max: 300000.0
  }
}
feature {
  name: "Company"
  type: BYTES
}
feature {
  name: "TypeName"
  type: BYTES
}
feature {
  name: "Inches"
  type: FLOAT
  float_domain {
    min: 10.0
    max: 32.0
  }
}
feature {
  name: "ScreenResolution"
  type: BYTES
}
feature {
  name: "Ram"
  type: FLOAT
  float_domain {
    min: 1.0
    max: 64.0
  }
}
feature {
  name: "Memory"
  type: BYTES
}
feature {
  name: "OpSys"
  type: BYTES
}
feature {
  name: "Weight"
  type: FLOAT
  float_domain {
    min: 0.5
    max: 5.0
  }
}
feature {
  name: "Cpu"
  type: BYTES
}
feature {
  name: "Gpu"
  type: BYTES
}

In [128]:
# 2️⃣ Generate statistics for each split
train_stats = tfdv.generate_statistics_from_dataframe(train_df)
eval_stats = tfdv.generate_statistics_from_dataframe(eval_df)
test_stats = tfdv.generate_statistics_from_dataframe(test_df)

In [129]:
# 3️⃣ Validate each split against the manual schema
train_anomalies = tfdv.validate_statistics(train_stats, schema)
eval_anomalies = tfdv.validate_statistics(eval_stats, schema)
test_anomalies = tfdv.validate_statistics(test_stats, schema)

In [130]:
# 4️⃣ Display the anomalies
print('🚀 Train anomalies:')
tfdv.display_anomalies(train_anomalies)

🚀 Train anomalies:


,Anomaly short description,Anomaly long description
Feature name,,
'Price',Multiple errors,Unexpectedly low values: -500<10000(upto six significant digits) Unexpectedly high value: 999999>300000(upto six significant digits)
'Inches',Out-of-range values,Unexpectedly high value: 35.6>32(upto six significant digits)
'__index_level_0__',New column,New column (column in data but not in schema)
'Weight',Multiple errors,Unexpectedly low values: 0.0002<0.5(upto six significant digits) Unexpectedly high value: 11.1>5(upto six significant digits)


In [131]:
print('🚀 Eval anomalies:')
tfdv.display_anomalies(eval_anomalies)

🚀 Eval anomalies:


,Anomaly short description,Anomaly long description
Feature name,,
'Price',Multiple errors,Unexpectedly low values: -500<10000(upto six significant digits) Unexpectedly high value: 999999>300000(upto six significant digits)
'Weight',Multiple errors,Unexpectedly low values: 0.0002<0.5(upto six significant digits) Unexpectedly high value: 11.1>5(upto six significant digits)
'__index_level_0__',New column,New column (column in data but not in schema)
'Inches',Out-of-range values,Unexpectedly high value: 35.6>32(upto six significant digits)


In [132]:
print('🚀 Test anomalies:')
tfdv.display_anomalies(test_anomalies)

🚀 Test anomalies:


,Anomaly short description,Anomaly long description
Feature name,,
'Inches',Out-of-range values,Unexpectedly high value: 35.6>32(upto six significant digits)
'Weight',Multiple errors,Unexpectedly low values: 0.0002<0.5(upto six significant digits) Unexpectedly high value: 11.1>5(upto six significant digits)
'__index_level_0__',New column,New column (column in data but not in schema)
'Price',Multiple errors,Unexpectedly low values: -500<10000(upto six significant digits) Unexpectedly high value: 999999>300000(upto six significant digits)


In [133]:
from tensorflow_metadata.proto.v0 import anomalies_pb2

def clean_df_with_schema(df, schema):
    stats = tfdv.generate_statistics_from_dataframe(df)
    anomalies = tfdv.validate_statistics(stats, schema)

    for feature_name, anomaly_info in anomalies.anomaly_info.items():
        print(f"\n=== {feature_name} ===")
        #print(anomaly_info)  # full proto contents
        feature_schema = next((f for f in schema.feature if f.name == feature_name), None)
        #print(feature_schema)

        for reason in anomaly_info.reason:
            short_desc = reason.short_description.lower()

            if "new column" in short_desc:
                if feature_name in df.columns:
                    print(f"🧹 Dropping new column: {feature_name}")
                    df = df.drop(columns=[feature_name])

            elif "out-of-range" in short_desc and feature_schema and feature_schema.float_domain:
                min_expected = feature_schema.float_domain.min
                max_expected = feature_schema.float_domain.max
                print(f"🧹 Clipping {feature_name} to [{min_expected}, {max_expected}]")
                df.loc[df[feature_name] < min_expected, feature_name] = min_expected
                df.loc[df[feature_name] > max_expected, feature_name] = max_expected
            else:
                print(reason)

    return df


In [134]:
clean_df_with_schema(train_df, schema)


=== Weight ===
🧹 Clipping Weight to [0.5, 5.0]
🧹 Clipping Weight to [0.5, 5.0]

=== Price ===
🧹 Clipping Price to [10000.0, 300000.0]
🧹 Clipping Price to [10000.0, 300000.0]

=== __index_level_0__ ===

=== Inches ===
🧹 Clipping Inches to [10.0, 32.0]


,Company,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price
987231,HP,Notebook,13.3,Full HD 1920x1080,Intel Core i7 7500U 2.7GHz,8.000000,256GB SSD,Intel HD Graphics 620,Windows 10,1.49,58075.2000
79954,Dell,Notebook,15.6,1366x768,Intel Celeron Dual Core N3060 1.60GHz,4.000000,500GB HDD,Intel HD Graphics 400,Windows 10,1.80,16463.5200
567130,unknown,unknown,15.6,unknown,NaN,8.467997,unknown,NaN,unknown,2.20,52054.5600
500891,Asus,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,4.000000,1TB HDD,Nvidia GeForce 920,Linux,2.10,27652.3200
55399,HP,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8.000000,256GB SSD,Intel HD Graphics 620,Windows 10,1.96,31914.1872
...,...,...,...,...,...,...,...,...,...,...,...
951873,HP,Notebook,15.6,1366x768,Intel Core i5 7200U 2.5GHz,4.000000,500GB HDD,Intel HD Graphics 620,Windows 10,1.86,25840.8000
350388,Dell,2 in 1 Convertible,15.6,Full HD / Touchscreen 1920x1080,Intel Core i7 8550U 1.8GHz,8.000000,256GB SSD,Intel UHD Graphics 620,Windows 10,1.56,55890.7200
628199,HP,Notebook,15.6,1366x768,Intel Core i3 6006U 2GHz,4.000000,500GB HDD,Intel HD Graphics 520,Windows 10,1.86,23389.9200
168290,Asus,Gaming,17.3,Full HD 1920x1080,AMD Ryzen 1600 3.2GHz,8.000000,256GB SSD + 1TB HDD,AMD Radeon RX 580,Windows 10,3.20,90309.6000


In [135]:
clean_df_with_schema(eval_df, schema)


=== Weight ===
🧹 Clipping Weight to [0.5, 5.0]
🧹 Clipping Weight to [0.5, 5.0]

=== __index_level_0__ ===

=== Inches ===
🧹 Clipping Inches to [10.0, 32.0]

=== Price ===
🧹 Clipping Price to [10000.0, 300000.0]
🧹 Clipping Price to [10000.0, 300000.0]


,Company,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price
599493,Toshiba,Ultrabook,14.0,IPS Panel Full HD 1920x1080,Intel Core i7 6600U 2.6GHz,16.000000,512GB SSD,Nvidia GeForce 930M,Windows 10,1.47,93985.9200
360869,Lenovo,Notebook,15.6,IPS Panel Full HD 1920x1080,Intel Core i5 6300HQ 2.3GHz,8.467997,1TB HDD,Nvidia GeForce GTX 950M,Windows 10,2.30,52054.5600
791508,Acer,Notebook,15.6,1366x768,Intel Celeron Dual Core 3205U 1.5GHz,2.000000,16GB SSD,Intel HD Graphics,Chrome OS,2.19,10602.7200
326428,Dell,Netbook,11.6,1366x768,Intel Pentium Quad Core N4200 1.1GHz,4.000000,128GB SSD,Intel HD Graphics 505,Windows 10,1.63,39640.3200
149121,Samsung,2 in 1 Convertible,15.0,Full HD / Touchscreen 1920x1080,Intel Core i7 7500U 2.7GHz,16.000000,256GB SSD,AMD Radeon 540,Windows 10,1.71,95850.7200
...,...,...,...,...,...,...,...,...,...,...,...
38067,Dell,Notebook,15.6,Full HD 1920x1080,Intel Core i7 7600U 2.8GHz,16.000000,256GB SSD,Nvidia GeForce 930MX,Windows 10,1.93,81912.1392
419241,HP,Notebook,15.6,Full HD 1920x1080,Intel Core i3 6006U 2GHz,4.000000,500GB HDD,Intel HD Graphics 520,No OS,1.86,18381.0672
591598,Lenovo,Notebook,14.0,1366x768,Intel Celeron Dual Core N3350 1.1GHz,4.000000,32GB Flash Storage,NaN,Windows 10,1.44,15557.7600
770227,Google,Ultrabook,12.3,Touchscreen 2400x1600,Intel Core i5 7Y57 1.2GHz,8.000000,128GB SSD,Intel HD Graphics 615,Chrome OS,1.10,67932.0000


In [136]:
clean_df_with_schema(test_df, schema)


=== Weight ===
🧹 Clipping Weight to [0.5, 5.0]
🧹 Clipping Weight to [0.5, 5.0]

=== Inches ===
🧹 Clipping Inches to [10.0, 32.0]

=== Price ===
🧹 Clipping Price to [10000.0, 300000.0]
🧹 Clipping Price to [10000.0, 300000.0]

=== __index_level_0__ ===


,Company,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price
8,MSI,Gaming,15.6,Full HD 1920x1080,Intel Core i7 6700HQ 2.6GHz,8.0,128GB SSD + 1TB HDD,Nvidia GeForce GTX 960M,Windows 10,2.300,62284.3200
13,Dell,Notebook,15.6,1366x768,Intel Core i5 7200U 2.5GHz,8.0,1TB HDD,AMD Radeon R7 M445,Windows 10,2.360,34045.3872
21,HP,Notebook,15.6,IPS Panel Full HD 1920x1080,Intel Core i7 6700HQ 2.6GHz,6.0,1TB HDD,Nvidia GeForce GTX 960M,Windows 10,2.180,42570.7200
27,Asus,Notebook,15.6,IPS Panel 4K Ultra HD 3840x2160,Intel Core i7 6700HQ 2.6GHz,12.0,128GB SSD + 1TB HDD,Intel HD Graphics 530,Windows 10,2.060,69210.7200
37,Dell,Notebook,15.6,1366x768,Intel Core i7 7500U 2.7GHz,8.0,1TB HDD,AMD Radeon R5 M430,Linux,2.300,42943.1472
...,...,...,...,...,...,...,...,...,...,...,...
999949,Toshiba,Notebook,13.3,IPS Panel Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8.0,256GB SSD,Intel HD Graphics 620,Windows 10,1.050,89084.1600
999969,Toshiba,Notebook,15.6,1366x768,Intel Core i5 6200U 2.3GHz,4.0,500GB HDD,Intel HD Graphics 520,Windows 10,2.100,41558.4000
999970,Microsoft,Ultrabook,13.5,Touchscreen 2256x1504,Intel Core i5 7200U 2.5GHz,8.0,256GB SSD,Intel HD Graphics 620,Windows 10 S,1.252,71395.2000
999976,Dell,2 in 1 Convertible,15.6,IPS Panel Full HD / Touchscreen 1920x1080,Intel Core i7 7500U 2.7GHz,12.0,512GB SSD,Intel HD Graphics 620,Windows 10,2.190,69210.7200


In [137]:
train_df.to_csv("dataset/train_cleaned.csv", index=False)
eval_df.to_csv("dataset/eval_cleaned.csv", index=False)
test_df.to_csv("dataset/test_cleaned.csv", index=False)

In [138]:
for dataset in ["train", "eval", "test"]:
    df = pd.read_csv(f'dataset/{dataset}_cleaned.csv')
    numeric_df = df.select_dtypes(include=[np.number])

    summary = pd.DataFrame({
        'min': numeric_df.min(),
        'max': numeric_df.max(),
        'mean': numeric_df.mean(),
        'median': numeric_df.median()
    })
    print(summary)

            min       max          mean    median
Inches     10.1      32.0     15.134918     15.60
Ram         1.0      64.0      8.465588      8.00
Weight      0.5       5.0      2.066989      2.06
Price   10000.0  300000.0  60181.385534  52054.56
            min       max          mean    median
Inches     10.1      32.0     15.130932     15.60
Ram         1.0      64.0      8.505412      8.00
Weight      0.5       5.0      2.066927      2.06
Price   10000.0  300000.0  60325.038014  52054.56
            min       max          mean    median
Inches     10.1      32.0     15.128757     15.60
Ram         1.0      64.0      8.449853      8.00
Weight      0.5       5.0      2.064327      2.06
Price   10000.0  300000.0  59937.349009  52054.56
